In [1]:
"""
Validação do Data Lake S3
Confirma que os dois tipos de dados estão presentes e corretos.
Corre numa célula do Jupyter.
"""
import boto3
import pandas as pd

s3 = boto3.client(
    's3',
    endpoint_url='http://localstack:4566',
    aws_access_key_id='test',
    aws_secret_access_key='test'
)

BUCKET = 'forest-risk-datalake'

def listar_parquets(prefixo):
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefixo)
    return [o for o in resp.get('Contents', []) if o['Key'].endswith('.parquet')]

def tamanho_total(ficheiros):
    return sum(o['Size'] for o in ficheiros) / 1024  # KB

print("=" * 60)
print("VALIDAÇÃO DO DATA LAKE — forest-risk-datalake")
print("=" * 60)

# ── 1. Dados históricos (CSV NASA) ────────────────────────────────
historico = listar_parquets('hotspots/')
print(f"\n📂 HISTÓRICO NASA FIRMS (hotspots/)")
print(f"   Ficheiros Parquet : {len(historico)}")
print(f"   Tamanho total     : {tamanho_total(historico):.1f} KB")

# Conta anos presentes
anos = set()
zonas = set()
for o in historico:
    partes = o['Key'].split('/')
    for p in partes:
        if p.startswith('ano='): anos.add(p.replace('ano=', ''))
        if p.startswith('grid_id='): zonas.add(p.replace('grid_id=', ''))
print(f"   Anos cobertos     : {sorted(anos)}")
print(f"   Zonas cobertas    : {len(zonas)} zonas")

# ── 2. Dados streaming (Spark) ────────────────────────────────────
streaming = listar_parquets('agregados_streaming/')
# remove metadata
streaming = [o for o in streaming if not '_spark_metadata' in o['Key']]
print(f"\n📡 STREAMING SPARK (agregados_streaming/)")
print(f"   Ficheiros Parquet : {len(streaming)}")
print(f"   Tamanho total     : {tamanho_total(streaming):.1f} KB")

if streaming:
    # Lê um ficheiro para confirmar o conteúdo
    obj = s3.get_object(Bucket=BUCKET, Key=streaming[0]['Key'])
    import io
    df = pd.read_parquet(io.BytesIO(obj['Body'].read()))
    print(f"   Colunas           : {list(df.columns)}")
    print(f"   Linhas na amostra : {len(df)}")
    print(f"   Zonas na amostra  : {df['grid_id'].unique().tolist()}")

# ── 3. Resumo final ───────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("RESUMO")
print(f"{'=' * 60}")
print(f"  Histórico NASA  : {'✅ OK' if len(historico) > 0 else '❌ VAZIO'} ({len(historico)} ficheiros)")
print(f"  Streaming Spark : {'✅ OK' if len(streaming) > 0 else '⏳ A aguardar janelas fecharem'} ({len(streaming)} ficheiros)")
estado = "✅ DATA LAKE COMPLETO" if len(historico) > 0 and len(streaming) > 0 else "⏳ INCOMPLETO"
print(f"\n  {estado}")
print("=" * 60)

VALIDAÇÃO DO DATA LAKE — forest-risk-datalake

📂 HISTÓRICO NASA FIRMS (hotspots/)
   Ficheiros Parquet : 578
   Tamanho total     : 6878.1 KB
   Anos cobertos     : ['2020', '2021', '2022', '2023', '2024']
   Zonas cobertas    : 10 zonas

📡 STREAMING SPARK (agregados_streaming/)
   Ficheiros Parquet : 4
   Tamanho total     : 4.1 KB
   Colunas           : ['janela_inicio', 'janela_fim', 'grid_id', 'n_leituras', 'risk_medio', 'risk_maximo', 'temp_media', 'humidade_media', 'vento_medio']
   Linhas na amostra : 0
   Zonas na amostra  : []

RESUMO
  Histórico NASA  : ✅ OK (578 ficheiros)
  Streaming Spark : ✅ OK (4 ficheiros)

  ✅ DATA LAKE COMPLETO
